# Signature Verification — PyTorch Implementation

This notebook is a **PyTorch port** of the original TensorFlow/Keras Siamese Network
(`Signature_Verification_model_training.ipynb`).

It trains a **Siamese Neural Network** to verify whether two handwritten signatures
belong to the same person (genuine) or one is forged.

---

## How a Siamese Network Works

A Siamese Network has **two identical branches** (sharing the same weights) that each
process one input image. The outputs (embedding vectors) are compared using a distance
metric. If the distance is small → same person (genuine). If large → forged.

```
Anchor Image   ──┐
                 ├── Shared CNN Embedding ──┐
Validation Img ──┘                          ├── L1 Distance ── Classifier ── P(genuine)
                                             └──────────────────────────────────────────
```

---

> **Original note:** The datasets are stored in Google Drive and this notebook was
> originally written for Google Colab. Paths use `/content/` as the working directory.

## 1. Import Libraries

We replace all TensorFlow/Keras imports with their PyTorch equivalents:

| TensorFlow / Keras | PyTorch Equivalent |
|---|---|
| `tensorflow as tf` | `torch`, `torch.nn`, `torch.optim` |
| `tf.data.Dataset` | `torch.utils.data.Dataset` + `DataLoader` |
| `Conv2D`, `Dense`, `Layer` | `nn.Conv2d`, `nn.Linear`, `nn.Module` |
| `tf.io.read_file` + `tf.io.decode_jpeg` | `PIL.Image.open()` |
| `tf.image.resize` | `torchvision.transforms.Resize` |
| `tf.keras.losses.BinaryFocalCrossentropy` | `torchvision.ops.sigmoid_focal_loss` |
| `tf.train.Checkpoint` | `torch.save(state_dict, path)` |
| `Precision`, `Recall` (Keras) | `sklearn.metrics.precision_score/recall_score` |

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# IMPORTS
# ─────────────────────────────────────────────────────────────────────────────

# Standard library — file and path management
import os
import shutil
import pathlib
import tarfile
import zipfile

# Numerical and tabular data
import pandas as pd
import numpy as np

# Image visualisation
import matplotlib.pyplot as plt

# ── PyTorch core ──────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# ── torchvision ───────────────────────────────────────────────────────────────
import torchvision.transforms as transforms
from torchvision.ops import sigmoid_focal_loss  # handles class imbalance

# ── Pillow (PIL) — lightweight image I/O replacing tf.io ─────────────────────
from PIL import Image

# ── Evaluation metrics ────────────────────────────────────────────────────────
from sklearn.metrics import precision_score, recall_score

# ── Reproducibility: fix random seeds for consistent results ─────────────────
torch.manual_seed(42)
np.random.seed(42)

# ── Device selection: use GPU if available, else CPU ─────────────────────────
# Training on GPU is ~10-50x faster for CNNs. This line auto-detects the device.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

---

## 2. Data Cleanup

Remove any previously extracted dataset folders to ensure a clean run.
This avoids stale files from prior executions causing path mismatches or
duplicate pairs being added to the dataset.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Remove previously extracted dataset folders (if they exist)
# Replaces the original: !rm -r /content/Dutch_Training_2011 ...
# Using Python's shutil.rmtree instead of shell commands for portability.
# ─────────────────────────────────────────────────────────────────────────────

CLEANUP_DIRS = [
    "/content/Dutch_Training_2011",
    "/content/NISDCC-offline-all-001-051-6g",
    "/content/signatures",
]

for d in CLEANUP_DIRS:
    if os.path.exists(d):
        shutil.rmtree(d)
        print(f"Removed: {d}")
    else:
        print(f"Not found (skipping): {d}")

---

## 3. Image Preprocessing

Each signature image goes through a **3-step preprocessing pipeline** before
being fed into the neural network:

1. **Grayscale conversion** — Signatures have no meaningful colour information.
   We load each image as a single-channel (L) grayscale image, reducing
   memory usage and focusing the model on shape, not colour.

2. **Resize to (100 × 200)** — All images must have the same dimensions for
   batch processing. We use `(height=100, width=200)` matching the original.

3. **Normalise to [0, 1]** — `transforms.ToTensor()` divides pixel values
   (0–255) by 255, producing floats in [0, 1]. It also re-orders axes from
   PIL's `(H, W)` to PyTorch's `(C, H, W)` convention.

> **PyTorch axis convention:** Unlike TensorFlow's `(H, W, C)`, PyTorch
> expects images as `(Channels, Height, Width)`. `ToTensor()` handles this.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Image dimensions — must match the embedding model's expected input
# (100 rows × 200 columns, matching INPUT_SHAPE=(100,200) in the original)
# ─────────────────────────────────────────────────────────────────────────────
INPUT_HEIGHT = 100   # number of pixel rows
INPUT_WIDTH  = 200   # number of pixel columns
INPUT_SHAPE  = (INPUT_HEIGHT, INPUT_WIDTH)

# ─────────────────────────────────────────────────────────────────────────────
# Transform pipeline
#
# transforms.Compose applies each transform sequentially:
#   1. Resize(INPUT_SHAPE)  — rescale to exactly (100, 200)
#   2. ToTensor()           — PIL (H,W) uint8 → torch.Tensor (1,H,W) float32 in [0,1]
#
# This replaces:
#   img = tf.image.resize(img, INPUT_SHAPE)
#   img = img / 255.0
# ─────────────────────────────────────────────────────────────────────────────
IMAGE_TRANSFORM = transforms.Compose([
    transforms.Resize(INPUT_SHAPE),   # (height, width) tuple for Resize
    transforms.ToTensor(),            # normalises to [0,1] and adds channel dim
])


def preprocess_image(image_path: str) -> torch.Tensor:
    """
    Load a single signature image from disk and apply the preprocessing pipeline.

    Replaces `def preprocess_image(image_path)` from the original TF notebook.

    Parameters
    ----------
    image_path : str
        Absolute or relative path to the image file (JPEG or PNG).

    Returns
    -------
    torch.Tensor
        Shape: (1, INPUT_HEIGHT, INPUT_WIDTH)
        Single-channel float32 tensor with pixel values in [0, 1].
    """
    # .convert("L") loads as grayscale (1 channel)
    # This matches decode_jpeg(byte_img, channels=1) in the original
    img = Image.open(image_path).convert("L")
    return IMAGE_TRANSFORM(img)   # returns tensor of shape (1, H, W)

---

## 4. Dataset Construction

We combine two signature datasets:

| Dataset | Source | Signers |
|---|---|---|
| **ICDAR 2009** — NISDCC | `NISDCC-offline-all-001-051-6g.tgz` | 51 |
| **ICDAR 2011** — Dutch Offline | `Dutch_Training_2011.zip` | varies |

For each dataset we build **(anchor, validation, label)** pairs:
- **Genuine–Genuine** → label `1` (same person, both directions)
- **Genuine–Forged** → label `0`
- **Forged–Forged** → *skipped* (ambiguous ground truth)

### Why pairs?
The Siamese network learns a *similarity function*, not a classification function.
It needs to see both matching and non-matching pairs during training.

### 4a. ICDAR 2009 — NISDCC Dataset

**Filename convention:** `NISDCC-<simulator>_<primary>_<count>_6g.PNG`

- `simulator` = ID of the person who *wrote* the signature
- `primary` = ID of the person whose signature is being *imitated*
- If `simulator == primary` → **genuine** signature
- If `simulator != primary` → **forged** (the forger is `simulator`)

We first reorganise images into a cleaner folder structure:
```
/content/signatures/<primary>/         # genuine signatures
/content/signatures/<primary>_forg/    # forged signatures
```

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Extract the ICDAR 2009 archive from Google Drive
# Replaces: !tar -xzf /content/drive/MyDrive/NISDCC-offline-all-001-051-6g.tgz -C /content/
# Using Python's tarfile module instead of shell commands for portability.
# ─────────────────────────────────────────────────────────────────────────────
FOLDER_2009       = "NISDCC-offline-all-001-051-6g"
ARCHIVE_PATH_2009 = f"/content/drive/MyDrive/{FOLDER_2009}.tgz"
EXTRACT_DIR       = "/content"

# Remove any leftover extraction before re-extracting
if os.path.exists(f"{EXTRACT_DIR}/{FOLDER_2009}"):
    shutil.rmtree(f"{EXTRACT_DIR}/{FOLDER_2009}")

with tarfile.open(ARCHIVE_PATH_2009, "r:gz") as tar:
    tar.extractall(path=EXTRACT_DIR)
    print(f"Extracted → {EXTRACT_DIR}/{FOLDER_2009}")

# Collect all PNG paths in the extracted folder
data_dir_2009    = pathlib.Path(f"{EXTRACT_DIR}/{FOLDER_2009}")
all_images_2009  = sorted(map(str, data_dir_2009.glob("*.PNG")))
print(f"Total ICDAR 2009 images: {len(all_images_2009)}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Reorganise ICDAR 2009 images into the shared /content/signatures/ layout
#
# Genuine  → /content/signatures/<primary>/<count>.PNG
# Forged   → /content/signatures/<primary>_forg/<simulator>_<count>.PNG
#
# This mirrors the original loop that used shutil.copyfile.
# ─────────────────────────────────────────────────────────────────────────────
SIG_DIR = "/content/signatures"
os.makedirs(SIG_DIR, exist_ok=True)

for image_path in all_images_2009:
    image_name = os.path.basename(image_path)
    # Filename: NISDCC-<sim>_<pri>_<cnt>_6g.PNG
    # character indices are identical to the original notebook
    simulator  = image_name[7:10]
    primary    = image_name[11:14]
    count      = image_name[15:18]

    if simulator == primary:  # Genuine signature
        dest_dir = os.path.join(SIG_DIR, primary)
        os.makedirs(dest_dir, exist_ok=True)
        shutil.copyfile(image_path, os.path.join(dest_dir, f"{count}.PNG"))
    else:                     # Forged signature
        dest_dir = os.path.join(SIG_DIR, f"{primary}_forg")
        os.makedirs(dest_dir, exist_ok=True)
        shutil.copyfile(image_path, os.path.join(dest_dir, f"{simulator}_{count}.PNG"))

print("ICDAR 2009 images organised into /content/signatures/")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Initialise global pair lists
# These accumulate pairs from BOTH the 2009 and 2011 datasets.
# ─────────────────────────────────────────────────────────────────────────────
anchor_images:     list = []   # paths to anchor (reference) signatures
validation_images: list = []   # paths to validation signatures
labels:            list = []   # 1 = genuine match, 0 = forged

# ─────────────────────────────────────────────────────────────────────────────
# Build (anchor, validation, label) pairs for ICDAR 2009
#
# Logic (identical to original):
#   - Same signer (primary1 == primary2), both genuine  → label 1 (both directions)
#   - Same signer, one genuine + one forged            → label 0
#   - Same signer, both forged                         → skip
#   - Different signers                                → skip
# ─────────────────────────────────────────────────────────────────────────────
for i in range(len(all_images_2009)):
    for j in range(i, len(all_images_2009)):
        name1 = os.path.basename(all_images_2009[i])
        sim1, pri1, cnt1 = name1[7:10], name1[11:14], name1[15:18]

        name2 = os.path.basename(all_images_2009[j])
        sim2, pri2, cnt2 = name2[7:10], name2[11:14], name2[15:18]

        if pri1 != pri2:
            continue   # different signers — not useful for pair training

        is_genuine1 = (sim1 == pri1)
        is_genuine2 = (sim2 == pri2)

        if not is_genuine1 and not is_genuine2:
            continue   # both forged — skip (no clean ground truth)

        path1_genuine = f"{SIG_DIR}/{pri1}/{cnt1}.PNG"
        path1_forged  = f"{SIG_DIR}/{pri1}_forg/{sim1}_{cnt1}.PNG"
        path2_genuine = f"{SIG_DIR}/{pri2}/{cnt2}.PNG"
        path2_forged  = f"{SIG_DIR}/{pri2}_forg/{sim2}_{cnt2}.PNG"

        if is_genuine1 and is_genuine2:    # Genuine–Genuine pair
            # Add both directions so the network sees all orderings
            anchor_images     += [path1_genuine, path2_genuine]
            validation_images += [path2_genuine, path1_genuine]
            labels            += [1, 1]

        elif is_genuine1:                  # Image 1 genuine, Image 2 forged
            anchor_images.append(path1_genuine)
            validation_images.append(path2_forged)
            labels.append(0)

        else:                              # Image 2 genuine, Image 1 forged
            anchor_images.append(path2_genuine)
            validation_images.append(path1_forged)
            labels.append(0)

print(f"Pairs after ICDAR 2009: {len(labels)}  |  "
      f"genuine={labels.count(1)}  forged={labels.count(0)}")

# Count unique 2009 signers — needed to offset 2011 signer IDs
# (divides by 2 because each signer has a genuine folder AND a _forg folder)
signs_2009 = len(list(pathlib.Path(SIG_DIR).glob("*"))) // 2
print(f"Unique signers in ICDAR 2009: {signs_2009}")

### 4b. ICDAR 2011 — Dutch Offline Dataset

**Source:** `Dutch_Training_2011.zip` (Google Drive)

**Folder layout inside the zip:**
- `Offline Forgeries/*.png` — forged signatures
- `Offline Genuine/*.PNG`   — genuine signatures

**Filename conventions:**
- Genuine: `<primary><count>.<ext>` — e.g., `00101.PNG` → primary=`001`, count=`01`
- Forged:  `<forger_id><primary>_<count>.<ext>` — e.g., `0002001_01.png`

**Index offset:** To avoid signer ID collisions with the 2009 dataset in the shared
`/content/signatures/` folder, each 2011 signer index is offset by `signs_2009`.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Extract Dutch_Training_2011.zip from Google Drive
# Replaces: !unzip -q /content/drive/MyDrive/Dutch_Training_2011.zip -d /content/Dutch_Training_2011
# ─────────────────────────────────────────────────────────────────────────────
ARCHIVE_PATH_2011 = "/content/drive/MyDrive/Dutch_Training_2011.zip"
EXTRACT_DIR_2011  = "/content/Dutch_Training_2011"

if os.path.exists(EXTRACT_DIR_2011):
    shutil.rmtree(EXTRACT_DIR_2011)

with zipfile.ZipFile(ARCHIVE_PATH_2011, "r") as zf:
    zf.extractall(EXTRACT_DIR_2011)
    print(f"Extracted → {EXTRACT_DIR_2011}")

# Collect genuine and forged image paths
forgeries_2011 = sorted(map(str,
    pathlib.Path(EXTRACT_DIR_2011 + "/Offline Forgeries").glob("*.png")))
genuine_2011   = sorted(map(str,
    pathlib.Path(EXTRACT_DIR_2011 + "/Offline Genuine").glob("*.PNG")))

print(f"ICDAR 2011 — Genuine: {len(genuine_2011)}  |  Forged: {len(forgeries_2011)}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Helper: compute zero-padded signer index with 2009 offset
#
# ICDAR 2011 signer IDs start at 001. To avoid collision with 2009 signers
# in the shared /content/signatures/ folder, we add signs_2009 to each ID.
# ─────────────────────────────────────────────────────────────────────────────
def offset_index(primary_str: str) -> str:
    """Return zero-padded (width=3) signer index offset by signs_2009."""
    return str(int(primary_str) + signs_2009).zfill(3)


# ─────────────────────────────────────────────────────────────────────────────
# Copy 2011 forged images → /content/signatures/<offset_index>_forg/
# ─────────────────────────────────────────────────────────────────────────────
for image_path in forgeries_2011:
    name      = os.path.basename(image_path)
    forger_id = name[:4]    # first 4 chars = forger ID
    primary   = name[4:7]   # next 3 chars  = original signer
    count     = name[8:10]  # chars 8-10    = sample count
    index     = offset_index(primary)

    dest_dir = os.path.join(SIG_DIR, f"{index}_forg")
    os.makedirs(dest_dir, exist_ok=True)
    shutil.copyfile(image_path, os.path.join(dest_dir, f"{forger_id}_{count}.PNG"))

# ─────────────────────────────────────────────────────────────────────────────
# Copy 2011 genuine images → /content/signatures/<offset_index>/
# ─────────────────────────────────────────────────────────────────────────────
for image_path in genuine_2011:
    name    = os.path.basename(image_path)
    primary = name[:3]   # first 3 chars = signer ID
    count   = name[4:6]  # chars 4-6     = sample count
    index   = offset_index(primary)

    dest_dir = os.path.join(SIG_DIR, index)
    os.makedirs(dest_dir, exist_ok=True)
    shutil.copyfile(image_path, os.path.join(dest_dir, f"{count}.PNG"))

print("ICDAR 2011 images merged into /content/signatures/")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Build genuine–genuine pairs for ICDAR 2011
# Same logic as 2009: both directions added, same signer only.
# ─────────────────────────────────────────────────────────────────────────────
for i in range(len(genuine_2011)):
    for j in range(i, len(genuine_2011)):
        name1    = os.path.basename(genuine_2011[i])
        primary1 = name1[:3];  count1 = name1[4:6];  idx1 = offset_index(primary1)

        name2    = os.path.basename(genuine_2011[j])
        primary2 = name2[:3];  count2 = name2[4:6];  idx2 = offset_index(primary2)

        if primary1 != primary2:
            continue   # different signers

        # Both directions → the network sees (A,B,1) and (B,A,1)
        anchor_images     += [f"{SIG_DIR}/{idx1}/{count1}.PNG",
                               f"{SIG_DIR}/{idx2}/{count2}.PNG"]
        validation_images += [f"{SIG_DIR}/{idx2}/{count2}.PNG",
                               f"{SIG_DIR}/{idx1}/{count1}.PNG"]
        labels            += [1, 1]

# ─────────────────────────────────────────────────────────────────────────────
# Build genuine–forged pairs for ICDAR 2011
# ─────────────────────────────────────────────────────────────────────────────
for genuine_path in genuine_2011:
    name1    = os.path.basename(genuine_path)
    primary1 = name1[:3];  count1 = name1[4:6];  idx1 = offset_index(primary1)

    for forged_path in forgeries_2011:
        name2      = os.path.basename(forged_path)
        forger_id2 = name2[:4];  primary2 = name2[4:7]
        count2     = name2[8:10];  idx2    = offset_index(primary2)

        if primary1 != primary2:
            continue   # different signers

        anchor_images.append(f"{SIG_DIR}/{idx1}/{count1}.PNG")
        validation_images.append(f"{SIG_DIR}/{idx2}_forg/{forger_id2}_{count2}.PNG")
        labels.append(0)

print(f"Total pairs (2009 + 2011): {len(labels)}  |  "
      f"genuine={labels.count(1)}  forged={labels.count(0)}")

---

## 5. DataFrame Overview

Each row in the DataFrame represents **one pair to be classified**:

| Column | Description |
|---|---|
| `anchor_images` | Path to the **reference** (anchor) signature |
| `validation_images` | Path to the signature being **verified** |
| `labels` | `1.0` = genuine (same person), `0.0` = forged |

### Class Imbalance

This dataset is **highly imbalanced** — genuine pairs typically outnumber forged pairs.
We handle this with **Focal Loss** in training (see Section 7), which focuses
the model on hard-to-classify examples rather than easy, well-classified ones.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Convert lists to numpy arrays for efficient indexing during Dataset construction
# float32 for labels because PyTorch loss functions expect float tensors
# ─────────────────────────────────────────────────────────────────────────────
anchor_images     = np.array(anchor_images)
validation_images = np.array(validation_images)
labels            = np.array(labels, dtype=np.float32)

# Build a summary DataFrame for visual inspection
df = pd.DataFrame({
    "anchor_images":     anchor_images,
    "validation_images": validation_images,
    "labels":            labels,
})

print(f"Dataset shape: {df.shape}")
display(df.head())
print("\nLabel distribution:")
display(df["labels"].value_counts())

---

## 6. PyTorch Dataset & DataLoader

In PyTorch, data pipelines are built with two components:

### `Dataset` (what to load)
A class that knows **where data lives** and **how to load one sample**.
We subclass `torch.utils.data.Dataset` and implement:
- `__len__()` — total number of samples
- `__getitem__(idx)` — load and return sample at index `idx`

This replaces `tf.data.Dataset.from_tensor_slices(...).map(preprocess_tuple)`.

### `DataLoader` (how to batch)
Wraps the Dataset and yields **batches** automatically.
- `shuffle=True` → randomise order each epoch (replaces `.shuffle(1024)`)
- `batch_size=16` → group 16 samples per batch (replaces `.batch(16)`)
- `num_workers=4` → parallel image loading (replaces `.prefetch(8)`)

In [ ]:
class SignaturePairDataset(Dataset):
    """
    PyTorch Dataset for signature verification pairs.

    Each sample is a triple:
        (anchor_tensor, validation_tensor, label)

    where tensors have shape (1, INPUT_HEIGHT, INPUT_WIDTH) and values in [0, 1].

    This is the PyTorch equivalent of:
        tf.data.Dataset.from_tensor_slices(...).map(preprocess_tuple)

    Parameters
    ----------
    anchor_paths : np.ndarray
        File paths for anchor (reference) signature images.
    val_paths : np.ndarray
        File paths for validation signature images.
    labels : np.ndarray
        Float32 array of labels — 1.0 = genuine, 0.0 = forged.
    transform : callable, optional
        torchvision transform pipeline applied to each image.
    """

    def __init__(
        self,
        anchor_paths: np.ndarray,
        val_paths: np.ndarray,
        labels: np.ndarray,
        transform=IMAGE_TRANSFORM,
    ):
        self.anchor_paths = anchor_paths
        self.val_paths    = val_paths
        self.labels       = labels
        self.transform    = transform

    def __len__(self) -> int:
        """Total number of pairs in this dataset split."""
        return len(self.labels)

    def __getitem__(self, idx: int):
        """
        Load and preprocess one (anchor, validation, label) triple.

        Called automatically by DataLoader for each sample in a batch.

        Returns
        -------
        anchor_tensor     : torch.Tensor  shape (1, H, W)   float32 in [0,1]
        validation_tensor : torch.Tensor  shape (1, H, W)   float32 in [0,1]
        label             : torch.Tensor  scalar float32    0.0 or 1.0
        """
        # Load images as grayscale PIL images, then apply transforms
        anchor_img = Image.open(self.anchor_paths[idx]).convert("L")
        val_img    = Image.open(self.val_paths[idx]).convert("L")

        anchor_tensor = self.transform(anchor_img)
        val_tensor    = self.transform(val_img)

        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        return anchor_tensor, val_tensor, label

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Train / Test split — 70% train, 30% test  (matches original notebook)
#
# We shuffle before splitting (via np.random.permutation) to ensure the split
# is random, not determined by the order pairs were added to the list.
# ─────────────────────────────────────────────────────────────────────────────
BATCH_SIZE  = 16      # 16 pairs per batch — same as original
TRAIN_RATIO = 0.70    # 70% training, 30% test

n_total  = len(labels)
n_train  = round(n_total * TRAIN_RATIO)

# Create a random permutation of indices for the split
perm      = np.random.permutation(n_total)
train_idx = perm[:n_train]
test_idx  = perm[n_train:]

# Build Dataset objects for each split
train_dataset = SignaturePairDataset(
    anchor_paths = anchor_images[train_idx],
    val_paths    = validation_images[train_idx],
    labels       = labels[train_idx],
)
test_dataset = SignaturePairDataset(
    anchor_paths = anchor_images[test_idx],
    val_paths    = validation_images[test_idx],
    labels       = labels[test_idx],
)

# Wrap in DataLoaders for automatic batching, shuffling, and parallel loading
train_loader = DataLoader(
    train_dataset,
    batch_size  = BATCH_SIZE,
    shuffle     = True,                        # randomise order each epoch
    num_workers = 4,                           # parallel image loading
    pin_memory  = DEVICE.type == "cuda",       # faster CPU→GPU transfers
)
test_loader = DataLoader(
    test_dataset,
    batch_size  = BATCH_SIZE,
    shuffle     = False,                       # fixed order for reproducible eval
    num_workers = 4,
    pin_memory  = DEVICE.type == "cuda",
)

print(f"Train: {len(train_dataset)} samples  |  {len(train_loader)} batches")
print(f"Test : {len(test_dataset)} samples  |  {len(test_loader)} batches")

---

## 7. Building the Siamese Neural Network

The network has three parts:

1. **Embedding Model** — a shared CNN that maps a signature image → a 4096-dim vector
2. **L1 Distance Layer** — computes element-wise absolute difference between embeddings
3. **Siamese Network** — ties both branches together with the shared embedding

In PyTorch every learnable component is an `nn.Module` subclass with:
- `__init__()` — define all layers
- `forward()` — define how data flows through layers

This replaces Keras's functional API (`Input → Conv2D → ... → Model`).

### 7a. Embedding Model (Shared CNN)

The embedding model learns to convert a raw signature image into a compact
**feature vector** (embedding) of length 4096. Two signatures from the same
person should produce *similar* embeddings; forged signatures should produce
*dissimilar* ones.

**Architecture (identical to original):**

```
Input  (1, 100, 200)                       ← grayscale image
  │
  ├─ Conv2d(1→64,   kernel=10) + ReLU  →  MaxPool2d(kernel=64, stride=2, pad=31, ceil_mode=True)
  ├─ Conv2d(64→128, kernel=7)  + ReLU  →  MaxPool2d(kernel=64, stride=2, pad=31, ceil_mode=True)
  ├─ Conv2d(128→128,kernel=4)  + ReLU  →  MaxPool2d(kernel=64, stride=2, pad=31, ceil_mode=True)
  ├─ Conv2d(128→256,kernel=4)  + ReLU
  ├─ Flatten
  └─ Linear(N → 4096) + Sigmoid         ← embedding vector
```

> **Note on MaxPool2d padding — faithful Keras `'same'` replication:**
> The original uses `MaxPool2D(64, (2,2), padding='same')`. Keras `'same'` computes
> output size as `ceil(input / stride)`.
>
> A naive `padding=32` in PyTorch uses `floor((input + 64) / 2)`, which **diverges**
> from `ceil(input/2)` when the input is even-sized (which it becomes after the first
> conv). The correct PyTorch equivalent is `padding=31, ceil_mode=True`, which forces
> ceil-mode output-size computation — exactly matching Keras at every stage.
>
> **Flattened size N** is computed automatically via a dry-run on a dummy tensor.

In [ ]:
class EmbeddingModel(nn.Module):
    """
    Shared CNN embedding network.

    Converts a grayscale signature image (B, 1, H, W) into a
    4096-dimensional feature vector via four convolutional blocks
    followed by a fully-connected sigmoid layer.

    PyTorch equivalent of `make_embedding()` in the original TF notebook.
    """

    def __init__(self):
        super().__init__()

        # ── Block 1: 1 → 64 channels, 10×10 kernel ────────────────────────────
        # Captures coarse stroke features across a large receptive field
        self.conv1 = nn.Conv2d(1,   64,  kernel_size=10)
        self.pool1 = nn.MaxPool2d(kernel_size=64, stride=2, padding=31, ceil_mode=True)

        # ── Block 2: 64 → 128 channels, 7×7 kernel ────────────────────────────
        # Learns mid-level stroke patterns
        self.conv2 = nn.Conv2d(64,  128, kernel_size=7)
        self.pool2 = nn.MaxPool2d(kernel_size=64, stride=2, padding=31, ceil_mode=True)

        # ── Block 3: 128 → 128 channels, 4×4 kernel ───────────────────────────
        # Captures finer curvature and connection details
        self.conv3 = nn.Conv2d(128, 128, kernel_size=4)
        self.pool3 = nn.MaxPool2d(kernel_size=64, stride=2, padding=31, ceil_mode=True)

        # ── Block 4: 128 → 256 channels, 4×4 kernel (no pooling) ──────────────
        # High-level feature extraction — no pooling to retain spatial detail
        self.conv4 = nn.Conv2d(128, 256, kernel_size=4)

        self.relu    = nn.ReLU()
        self.flatten = nn.Flatten()

        # ── Compute flattened dimension via dry-run ────────────────────────────
        # Pass a zero tensor through the conv blocks to get the exact output size.
        # This avoids hardcoding a magic number that depends on kernel/stride math.
        with torch.no_grad():
            dummy = torch.zeros(1, 1, INPUT_HEIGHT, INPUT_WIDTH)
            dummy = self._forward_conv(dummy)
            self._flat_size = dummy.shape[1]

        # ── Fully-connected embedding head: N → 4096 ──────────────────────────
        # Sigmoid activation maps values to (0, 1), producing a bounded embedding.
        # Matches Dense(4096, activation='sigmoid') in the original.
        self.fc1     = nn.Linear(self._flat_size, 4096)
        self.sigmoid = nn.Sigmoid()

    def _forward_conv(self, x: torch.Tensor) -> torch.Tensor:
        """Internal helper — pass through all conv+pool blocks and flatten."""
        x = self.pool1(self.relu(self.conv1(x)))
        x = self.pool2(self.relu(self.conv2(x)))
        x = self.pool3(self.relu(self.conv3(x)))
        x = self.relu(self.conv4(x))
        return self.flatten(x)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ----------
        x : torch.Tensor  shape (B, 1, H, W)  — batch of grayscale signature images

        Returns
        -------
        torch.Tensor  shape (B, 4096)  — embedding vectors
        """
        x = self._forward_conv(x)
        return self.sigmoid(self.fc1(x))


# Instantiate and inspect the embedding model
embedding_model = EmbeddingModel().to(DEVICE)
print(f"Flattened size before FC layer: {embedding_model._flat_size}")
print(embedding_model)

### 7b. L1 Distance Layer

The **L1 (Manhattan) distance** between two vectors is the element-wise absolute
difference: `|a - b|`.

This is a simple, interpretable similarity measure: if two embeddings are identical
(same person), the distance is all zeros. A large distance means the signatures
look different to the network.

### 7c. Siamese Network (Full Model)

Both the anchor and validation images pass through the **same** `EmbeddingModel`
(weight sharing is natural in PyTorch — just call the same module twice in `forward()`).

The L1 distance between embeddings is then fed to a binary classifier:
```
P(genuine) = σ( W · |emb(anchor) − emb(validation)| + b )
```

In [ ]:
class L1Dist(nn.Module):
    """
    L1 (Manhattan) distance between two embedding vectors.

    Computes element-wise absolute difference: |embedding_anchor - embedding_val|

    PyTorch equivalent of the Keras custom `class L1Dist(Layer)` in the original notebook.
    Note: No trainable parameters — this is a pure distance computation.

    Input shapes
    ------------
    embedding_anchor     : torch.Tensor  shape (B, 4096)
    embedding_validation : torch.Tensor  shape (B, 4096)

    Output shape
    ------------
    torch.Tensor  shape (B, 4096) — element-wise |a - b|
    """

    def forward(
        self,
        embedding_anchor: torch.Tensor,
        embedding_validation: torch.Tensor,
    ) -> torch.Tensor:
        return torch.abs(embedding_anchor - embedding_validation)


class SiameseNetwork(nn.Module):
    """
    Full Siamese Network for signature verification.

    Architecture:
        anchor     ──┐
                     ├── EmbeddingModel (shared) ──┐
        validation ──┘                              ├── L1Dist ── Linear(4096,1) ── Sigmoid
                                                    └──────────────────────────────────────

    Weight sharing is implicit in PyTorch: calling `self.embedding(x)` twice
    uses the same weights for both branches (unlike Keras where you must
    explicitly share layer instances).

    PyTorch equivalent of `make_siamese_model()` in the original notebook.

    Parameters
    ----------
    embedding : EmbeddingModel
        The shared CNN embedding model (instantiated externally).
    """

    def __init__(self, embedding: EmbeddingModel):
        super().__init__()
        self.embedding   = embedding       # shared CNN — called twice in forward()
        self.l1_distance = L1Dist()
        self.classifier  = nn.Linear(4096, 1)  # maps 4096-dim distance → scalar
        self.sigmoid     = nn.Sigmoid()         # squash to probability [0, 1]

    def forward(
        self,
        anchor: torch.Tensor,
        validation: torch.Tensor,
    ) -> torch.Tensor:
        """
        Parameters
        ----------
        anchor     : torch.Tensor  shape (B, 1, H, W) — anchor signature images
        validation : torch.Tensor  shape (B, 1, H, W) — validation signature images

        Returns
        -------
        torch.Tensor  shape (B, 1) — P(genuine) probability in [0, 1]
        """
        # Both branches use the SAME embedding weights (Siamese architecture)
        emb_anchor = self.embedding(anchor)
        emb_val    = self.embedding(validation)

        # Compute L1 distance between embeddings
        distance = self.l1_distance(emb_anchor, emb_val)

        # Binary classification: distance → probability
        return self.sigmoid(self.classifier(distance))


# Instantiate the full Siamese network
siamese_model = SiameseNetwork(embedding=embedding_model).to(DEVICE)
print(siamese_model)

---

## 8. Training Setup

### Loss Function — Binary Focal Cross-Entropy

Plain Binary Cross-Entropy (BCE) would be dominated by the majority class
(genuine pairs). **Focal Loss** solves this by adding a modulating factor
`(1 - p_t)^gamma` that down-weights easy examples and focuses training on
the hard, misclassified ones.

```
FL(p_t) = -(1 - p_t)^gamma · log(p_t)
```

`torchvision.ops.sigmoid_focal_loss` provides this directly.

### Optimizer — Adam (lr = 1e-4)

Adam with `lr=1e-4` is identical to `tf.optimizers.Adam(1e-4)` in the original.

### Checkpointing

We save a checkpoint (model weights + optimizer state) every 10 epochs using
`torch.save`, replacing `tf.train.Checkpoint`.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Hyper-parameters
# ─────────────────────────────────────────────────────────────────────────────
LEARNING_RATE      = 1e-4   # same as original tf.optimizers.Adam(1e-4)
EPOCHS             = 50     # same as original EPOCHS = 50
CHECKPOINT_DIR     = "./training_checkpoints"
CHECKPOINT_EVERY_N = 10     # save checkpoint every N epochs (same as original)

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────────
# Optimizer — Adam
# Equivalent to tf.optimizers.Adam(1e-4) in the original
# ─────────────────────────────────────────────────────────────────────────────
optimizer = optim.Adam(siamese_model.parameters(), lr=LEARNING_RATE)

# ─────────────────────────────────────────────────────────────────────────────
# Focal Loss wrapper
#
# torchvision.ops.sigmoid_focal_loss expects raw logits (before sigmoid),
# but our SiameseNetwork already applies Sigmoid as the final activation.
#
# Solution: convert probability p back to logit space: logit = log(p / (1-p))
# We clamp p to avoid log(0) or log(1) numerical instability.
# ─────────────────────────────────────────────────────────────────────────────
def focal_loss(y_hat: torch.Tensor, y_true: torch.Tensor) -> torch.Tensor:
    """
    Compute Binary Focal Cross-Entropy loss.

    Equivalent to tf.keras.losses.BinaryFocalCrossentropy() in the original.

    Parameters
    ----------
    y_hat  : torch.Tensor  shape (B, 1)  — model output probabilities in [0, 1]
    y_true : torch.Tensor  shape (B,)    — ground-truth labels (0.0 or 1.0)

    Returns
    -------
    torch.Tensor  scalar loss value
    """
    # Clamp to (ε, 1-ε) to avoid log(0) — same stability trick used internally by TF
    p     = y_hat.squeeze(1).clamp(1e-7, 1 - 1e-7)
    # Inverse sigmoid: converts probability → logit for sigmoid_focal_loss
    logit = torch.log(p / (1.0 - p))
    return sigmoid_focal_loss(logit, y_true, reduction="mean")


print(f"Optimizer : Adam  (lr={LEARNING_RATE})")
print(f"Loss      : Binary Focal Cross-Entropy (torchvision.ops.sigmoid_focal_loss)")
print(f"Epochs    : {EPOCHS}")
print(f"Checkpoint: every {CHECKPOINT_EVERY_N} epochs → {CHECKPOINT_DIR}/")

---

## 9. Training

### How PyTorch Training Differs from TensorFlow

| TensorFlow | PyTorch |
|---|---|
| `tf.GradientTape()` records ops | Autograd is always tracking — no context manager needed |
| `tape.gradient(loss, variables)` | `loss.backward()` — computes gradients in-place |
| `optimizer.apply_gradients(...)` | `optimizer.step()` — applies computed gradients |
| `optimizer.zero_gradients()` (implicit) | **Must call `optimizer.zero_grad()` manually each step** |
| `@tf.function` JIT compilation | `torch.compile()` (optional, PyTorch 2.x) |

The key footgun in PyTorch is forgetting `optimizer.zero_grad()` — without it,
gradients **accumulate** across batches instead of being reset.

In [ ]:
def train_step(
    anchor: torch.Tensor,
    validation: torch.Tensor,
    y_true: torch.Tensor,
) -> float:
    """
    Perform one forward + backward pass on a single batch.

    PyTorch equivalent of `@tf.function def train_step(batch)` with
    `tf.GradientTape` from the original notebook.

    Steps:
        1. Zero out stale gradients from the previous batch
        2. Forward pass: compute predictions
        3. Compute focal loss against ground-truth labels
        4. Backward pass: compute gradients via autograd
        5. Optimizer step: update weights

    Parameters
    ----------
    anchor     : torch.Tensor  shape (B, 1, H, W)  — already on DEVICE
    validation : torch.Tensor  shape (B, 1, H, W)  — already on DEVICE
    y_true     : torch.Tensor  shape (B,)           — float32 labels on DEVICE

    Returns
    -------
    float — scalar loss for this batch
    """
    # Step 1: Clear gradients — CRITICAL in PyTorch, easy to forget
    optimizer.zero_grad()

    # Step 2: Forward pass — shape (B, 1)
    y_hat = siamese_model(anchor, validation)

    # Step 3: Compute focal loss
    loss = focal_loss(y_hat, y_true)

    # Step 4: Backward pass — PyTorch autograd computes ∂loss/∂weights
    loss.backward()

    # Step 5: Update weights using computed gradients
    optimizer.step()

    return loss.item()   # .item() converts scalar tensor → Python float

In [ ]:
def train(data_loader: DataLoader, epochs: int) -> list:
    """
    Full training loop — iterate over all epochs and batches.

    PyTorch equivalent of `def train(data, EPOCHS)` in the original notebook.
    Saves a checkpoint every `CHECKPOINT_EVERY_N` epochs.

    Parameters
    ----------
    data_loader : DataLoader  — yields (anchor, validation, label) batches
    epochs      : int         — total number of training epochs

    Returns
    -------
    list[float] — average loss per epoch (used for learning curve plot)
    """
    epoch_losses = []

    for epoch in range(1, epochs + 1):
        # Set model to training mode once per epoch (more idiomatic than per-batch)
        # This matters if Dropout or BatchNorm layers are added later.
        siamese_model.train()
        batch_losses = []
        print(f"\nEpoch {epoch}/{epochs}")

        for batch_idx, (anchor, validation, y_true) in enumerate(data_loader):
            # Move data to the selected device (GPU or CPU)
            # .to(DEVICE) is a no-op if already on the correct device
            anchor     = anchor.to(DEVICE)
            validation = validation.to(DEVICE)
            y_true     = y_true.to(DEVICE)

            loss = train_step(anchor, validation, y_true)
            batch_losses.append(loss)

            # Print progress every 50 batches (replaces Keras Progbar)
            if (batch_idx + 1) % 50 == 0 or (batch_idx + 1) == len(data_loader):
                avg_so_far = float(np.mean(batch_losses))
                print(
                    f"  Batch {batch_idx + 1:>5}/{len(data_loader)} "
                    f"| loss: {loss:.4f}  (epoch avg: {avg_so_far:.4f})"
                )

        avg_epoch_loss = float(np.mean(batch_losses))
        epoch_losses.append(avg_epoch_loss)
        print(f"  → Epoch {epoch} complete | avg loss: {avg_epoch_loss:.4f}")

        # Save a checkpoint every CHECKPOINT_EVERY_N epochs
        # torch.save replaces tf.train.Checkpoint.save()
        # We save both model weights AND optimizer state so training can be resumed.
        if epoch % CHECKPOINT_EVERY_N == 0:
            ckpt_path = os.path.join(CHECKPOINT_DIR, f"ckpt_epoch_{epoch}.pth")
            torch.save({
                "epoch":                epoch,
                "model_state_dict":     siamese_model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "loss":                 avg_epoch_loss,
            }, ckpt_path)
            print(f"  Checkpoint saved: {ckpt_path}")

    return epoch_losses

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Start training
# This cell will take a long time to run (50 epochs × all batches).
# Use a GPU runtime in Colab (Runtime → Change runtime type → GPU) for speed.
# ─────────────────────────────────────────────────────────────────────────────
epoch_losses = train(train_loader, EPOCHS)

---

## 10. Learning Curve

Plot the average training loss per epoch. A steadily decreasing curve indicates
the model is learning to distinguish genuine signatures from forged ones.

If the loss plateaus early, consider:
- Reducing the learning rate
- Adding dropout for regularisation
- Increasing model capacity

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(range(1, EPOCHS + 1), epoch_losses, marker="o", linewidth=1.5, markersize=4)
plt.title("Training Loss per Epoch", fontsize=14)
plt.xlabel("Epoch")
plt.ylabel("Focal Loss (avg)")
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig("training_loss.png", dpi=120)
plt.show()
print("Plot saved as training_loss.png")

---

## 11. Save & Load Model

### Saving

`torch.save(state_dict, path)` saves only the learned **weights** (not the full
Python object). This is smaller, more portable, and version-safe.

This replaces `model.save('siamesemodelfocal.h5')` from the original.

### Loading

To reload: **first define the architecture** (all class definitions must exist
in the session), then call `model.load_state_dict(torch.load(path))`.

Unlike Keras, no `custom_objects` dictionary is needed — the architecture
is defined in plain Python classes above.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Save trained model weights
# Replaces: model.save('siamesemodelfocal.h5')
# ─────────────────────────────────────────────────────────────────────────────
MODEL_SAVE_PATH = "siamese_model_focal.pth"

torch.save(siamese_model.state_dict(), MODEL_SAVE_PATH)
print(f"Model saved → {MODEL_SAVE_PATH}")

# Optional: also save to Google Drive for persistence across Colab sessions
# GDRIVE_SAVE_PATH = "/content/drive/MyDrive/siamese_model_focal.pth"
# torch.save(siamese_model.state_dict(), GDRIVE_SAVE_PATH)
# print(f"Model also saved → {GDRIVE_SAVE_PATH}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Load the saved model
# Replaces:
#   tf.keras.models.load_model('x.h5', custom_objects={'L1Dist':L1Dist, ...})
#
# In PyTorch, no custom_objects dict is needed — the class definitions above
# are sufficient. We reconstruct the architecture, then load the weights.
# ─────────────────────────────────────────────────────────────────────────────

# Step 1: Reconstruct the architecture (same class definitions as above)
loaded_embedding = EmbeddingModel().to(DEVICE)
loaded_model     = SiameseNetwork(embedding=loaded_embedding).to(DEVICE)

# Step 2: Load the saved weights into the model
# map_location ensures weights are loaded to the correct device (GPU/CPU)
loaded_model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=DEVICE))

# Step 3: Switch to evaluation mode
# .eval() disables dropout and batch normalisation training behaviour
# Always call this before inference!
loaded_model.eval()

print("Model loaded and ready for inference.")

---

## 12. Model Evaluation — Precision & Recall

We evaluate on the held-out **test set** using two metrics:

| Metric | Meaning |
|---|---|
| **Precision** | Of all pairs predicted genuine, what fraction truly are? |
| **Recall** | Of all genuinely matched pairs, what fraction did we correctly detect? |
| **F1 Score** | Harmonic mean of Precision and Recall — balanced metric |

A threshold of **0.5** on the sigmoid output converts probabilities to binary
predictions — identical to the original `1 if i > 0.5 else 0` logic.

`sklearn.metrics` replaces `tensorflow.keras.metrics.Precision/Recall`.

> **Note:** We wrap inference in `torch.no_grad()` to disable gradient
> computation. This saves memory and speeds up evaluation — no gradients
> are needed when we're just doing a forward pass.

In [ ]:
def evaluate(model: SiameseNetwork, data_loader: DataLoader):
    """
    Evaluate the Siamese Network on a DataLoader.

    PyTorch equivalent of `def evaluate(data)` in the original notebook.

    Parameters
    ----------
    model       : SiameseNetwork  — trained model (should be in eval mode)
    data_loader : DataLoader      — yields (anchor, validation, label) batches

    Returns
    -------
    tuple[float, float] — (precision, recall)
    """
    model.eval()   # disable dropout / batch-norm training behaviour

    all_y_true = []
    all_y_hat  = []

    # torch.no_grad() disables autograd — saves memory & time during inference
    with torch.no_grad():
        for anchor, validation, y_true in data_loader:
            anchor     = anchor.to(DEVICE)
            validation = validation.to(DEVICE)

            # Forward pass — shape (B, 1); .squeeze(1) → (B,)
            probs = model(anchor, validation).squeeze(1).cpu().numpy()

            # Apply threshold: probability > 0.5 → genuine (1), else forged (0)
            # Identical to: [1 if i > 0.5 else 0 for i in siamese_model.predict(...)]
            preds = (probs > 0.5).astype(int)

            all_y_true.extend(y_true.numpy().astype(int))
            all_y_hat.extend(preds)

    # sklearn replaces tf.keras.metrics.Precision / Recall
    precision = precision_score(all_y_true, all_y_hat, zero_division=0)
    recall    = recall_score(all_y_true, all_y_hat, zero_division=0)

    return precision, recall


# ─────────────────────────────────────────────────────────────────────────────
# Run evaluation on the test set
# ─────────────────────────────────────────────────────────────────────────────
precision, recall = evaluate(loaded_model, test_loader)

# Compute F1 score — harmonic mean of precision and recall
# Small epsilon (1e-8) avoids division by zero when both are 0
f1 = 2 * precision * recall / (precision + recall + 1e-8)

print(f"Test Precision : {precision:.4f}")
print(f"Test Recall    : {recall:.4f}")
print(f"F1 Score       : {f1:.4f}")